# 13F point-in-time data-spine audit

This notebook tests the provisional policies in `docs/13F_DATA_SPINE.md` against the complete local 13F history. It audits accession joins, amendment sequences, coverage boundaries, manager identifiers, instrument classes, issuer-identity coverage, mapping snapshots, and timestamp ordering.

It does **not** build the investor–asset matrix. Raw identifiers and individual holdings are never displayed. Executed outputs and aggregate caches remain local under ignored `data/` paths.

## Audit contract

The immutable grain remains manager CIK × filing accession × reported holding line. Economic dates, filing timestamps, availability timestamps, and later model-decision timestamps are separate.

The holdings scan reads every row in memory-bounded Parquet batches. Its aggregate results are cached using a fingerprint of all local partitions so unchanged inputs do not require another complete scan.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_spine_13f import (
    AUDIT_CONTRACT_VERSION,
    COVER_PAGE_AUDIT_COLUMNS,
    amendment_quality_summary,
    amendment_sequence_summary,
    amendment_type_summary,
    filing_cover_join_audit,
    full_history_holdings_audit,
    manager_identity_audit,
    mapping_snapshot_audit,
    parquet_input_fingerprint,
    prepare_amendment_events,
    quarter_coverage_audit,
)
from src.full_history_13f import (
    REQUIRED_FILING_COLUMNS,
    parquet_table_inventory,
    read_parquet_columns,
)


In [ ]:
PRODUCT_ROOT = PROJECT_ROOT / "data" / "full_history" / "13f" / "v1"
TABLES_ROOT = PRODUCT_ROOT / "tables"
FILINGS_DIR = TABLES_ROOT / "filings"
COVER_PAGES_DIR = TABLES_ROOT / "cover_pages"
HOLDINGS_DIR = TABLES_ROOT / "holdings"
IDENTITY_LINKS_DIR = TABLES_ROOT / "issuer_identity_links"
CACHE_DIR = PROJECT_ROOT / "data" / "audits" / "13f_data_spine"

if not TABLES_ROOT.is_dir():
    raise FileNotFoundError(
        "The ignored full-history 13F mirror is missing. Run the documented sync first."
    )
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Physical reconciliation

Footer totals provide an independent expected row count. The complete holdings scan below must reconcile to this inventory.

In [ ]:
inventory = parquet_table_inventory(PRODUCT_ROOT)
display(inventory)


## 2. Filing and cover-page accession joins

Amendment type and report type come from cover pages, while timing and holdings are linked by accession. Coverage and duplicate diagnostics must be reviewed before any join becomes production logic.

In [ ]:
filings_raw = read_parquet_columns(FILINGS_DIR, REQUIRED_FILING_COLUMNS)
cover_pages = read_parquet_columns(COVER_PAGES_DIR, COVER_PAGE_AUDIT_COLUMNS)

display(filing_cover_join_audit(filings_raw, cover_pages))


## 3. Amendment signals and sequences

`RESTATEMENT` and `NEW_HOLDINGS` are economically different. The first replaces the observable state; the second supplements it. This section parses all amendment metadata and reports conflicts, missing types, and manager-quarter event sequences without displaying managers.

In [ ]:
amendment_events = prepare_amendment_events(cover_pages)

display(amendment_quality_summary(amendment_events))
display(amendment_type_summary(amendment_events))
display(amendment_sequence_summary(amendment_events))


## 4. Continuous coverage and partial quarters

A quarter is eligible only when observation began by quarter-end and the local data freeze extends through the normal filing deadline. Missing manager filings remain unavailable rather than becoming zeros or forward-filled portfolios.

In [ ]:
coverage = quarter_coverage_audit(filings_raw)
display(coverage)

eligible_coverage = coverage.loc[coverage["eligible_complete_window"]]
display(
    pd.DataFrame(
        [
            {
                "first_eligible_quarter": eligible_coverage["report_date_utc"].min(),
                "last_eligible_quarter": eligible_coverage["report_date_utc"].max(),
                "eligible_quarters": len(eligible_coverage),
            }
        ]
    )
)


## 5. Manager identity stability

The baseline manager key is the zero-preserving CIK. Names and secondary regulatory identifiers are audited but never used for fuzzy automatic consolidation.

In [ ]:
display(manager_identity_audit(cover_pages))


## 6. Complete holdings scan

The following call scans every holdings row on the first run. It retains only quarterly aggregate diagnostics and reuses them when the complete partition fingerprint is unchanged.

The instrument classifier is deliberately provisional. `cash_share_candidate` means a share amount with no put/call flag; it is not proof of common stock, because the filing class label alone cannot reliably identify every ETF, fund, or special security.

In [ ]:
holdings_audit = full_history_holdings_audit(
    HOLDINGS_DIR,
    cache_directory=CACHE_DIR / "holdings",
    batch_size=250_000,
)

display(pd.DataFrame([holdings_audit.metadata]))

expected_holdings_rows = int(
    inventory.loc[inventory["table"].eq("holdings"), "rows"].iloc[0]
)
if int(holdings_audit.metadata["rows_scanned"]) != expected_holdings_rows:
    raise AssertionError("The complete holdings scan did not reconcile to footer totals")


### Instrument-channel coverage

Review both rows and reported value. A broad model must keep options, convertible or debt-like positions, funds, and cash-share candidates as typed channels rather than adding them as equivalent exposures.

In [ ]:
display(holdings_audit.instrument)


### Issuer-identity coverage

This is full-history coverage by quarter and declared confidence. Missing or low-confidence mappings remain explicit; current names and tickers are not fallback join keys.

In [ ]:
display(holdings_audit.issuer_identity)


### Holdings timestamp integrity

Timing checks are repeated over every holding, not inferred only from the smaller filing table.

In [ ]:
display(holdings_audit.timing)


## 7. Identity-mapping snapshots

Mapping snapshot dates are audited separately because a present-day issuer mapping cannot silently rewrite a historical asset identity. A production mapping join still requires row-level effective-date evidence.

In [ ]:
mapping_fingerprint = parquet_input_fingerprint(IDENTITY_LINKS_DIR)
mapping_cache = CACHE_DIR / (
    f"{AUDIT_CONTRACT_VERSION}-{mapping_fingerprint}-mapping-snapshot.parquet"
)

if mapping_cache.is_file():
    mapping_snapshots = pd.read_parquet(mapping_cache)
else:
    mapping_snapshots = mapping_snapshot_audit(IDENTITY_LINKS_DIR)
    mapping_snapshots.to_parquet(mapping_cache, index=False)

display(mapping_snapshots)


In [ ]:
mapping_dates = pd.to_datetime(
    mapping_snapshots["mapping_snapshot_date"],
    errors="coerce",
    utc=True,
)
first_model_cutoff = eligible_coverage["information_cutoff_utc"].min()
display(
    pd.DataFrame(
        [
            {
                "first_model_information_cutoff": first_model_cutoff,
                "earliest_mapping_snapshot": mapping_dates.min(),
                "mapping_rows_without_snapshot": int(
                    mapping_snapshots.loc[mapping_dates.isna(), "rows"].sum()
                ),
                "historical_snapshot_available_by_first_cutoff": bool(
                    mapping_dates.le(first_model_cutoff).any()
                ),
            }
        ]
    )
)


## Audit gate

The next investor–issuer matrix remains blocked until the aggregate evidence above is reviewed and the following rules are finalized:

1. amendment conflicts and sequences are either resolved or explicitly excluded;
2. the first and last eligible quarters are accepted;
3. a point-in-time security master separates common equity from funds and special instruments;
4. permitted issuer identity sources and confidence levels are fixed;
5. CIK continuity exceptions are documented with effective dates;
6. a real exchange calendar fixes information, decision, and execution timestamps;
7. every exclusion reports rows and value removed.

No model or backtest should begin by replacing these unresolved decisions with implicit defaults.